# Notebook 18 — Integrated DiveLab Capstone

**Companion to Chapter 18**

This capstone sends one synthetic depth history through the principal models developed across DiveLab. It introduces no new operational algorithm: its purpose is to reveal coupling, state, uncertainty, and different time scales.

> **Educational scope.** The profile and calculations are synthetic. This notebook is not a dive planner, decompression tool, gas-planning tool, training standard, or substitute for a validated dive computer.

## Learning objectives

- connect pressure, gas volume, buoyancy, gas demand, sensor estimation, and inert-gas states;
- distinguish common input history from subsystem state;
- visualize processes evolving on different time scales;
- identify assumptions and validation boundaries in an integrated model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize":(9,5),"axes.grid":True})
rho,g,p0=1025.0,9.80665,101325.0

def ambient_pressure_pa(depth_m):
    return p0+rho*g*np.asarray(depth_m)

def ambient_pressure_bar(depth_m):
    return ambient_pressure_pa(depth_m)/1e5

## 1. One synthetic depth history

The profile is designed only to exercise the models: descent, a changing-depth interval, ascent, and a long surface observation.

In [ ]:
def synthetic_depth(t_min):
    t=np.asarray(t_min)
    z=np.zeros_like(t,dtype=float)
    z=np.where(t<3,18*t/3,z)
    z=np.where((t>=3)&(t<15),18,z)
    z=np.where((t>=15)&(t<20),18-6*(t-15)/5,z)
    z=np.where((t>=20)&(t<25),12,z)
    z=np.where((t>=25)&(t<29),12*(29-t)/4,z)
    return np.maximum(z,0)

t=np.linspace(0,180,18001)
z=synthetic_depth(t)
p_bar=ambient_pressure_bar(z)

fig,ax=plt.subplots(2,1,sharex=True,figsize=(9,6))
ax[0].plot(t,z); ax[0].invert_yaxis(); ax[0].set_ylabel("Depth [m]")
ax[1].plot(t,p_bar); ax[1].set(xlabel="Time [min]",ylabel="Pressure [bar abs]")
plt.show()

## 2. Flexible gas volume and buoyancy contribution

A reference gas volume follows Boyle's law. Tidal lung-volume variation is added separately as a periodic buoyancy disturbance while submerged.

In [ ]:
v_surface=8e-3
v_gas=v_surface*p0/ambient_pressure_pa(z)
submerged=z>0.1
tidal=0.35e-3*np.sin(2*np.pi*0.25*60*t)*submerged
tidal_force=rho*g*tidal

fig,ax=plt.subplots(2,1,sharex=True,figsize=(9,6))
ax[0].plot(t,1e3*v_gas); ax[0].set_ylabel("Flexible gas [L]")
ax[1].plot(t,tidal_force); ax[1].set(xlabel="Time [min]",ylabel="Tidal buoyancy [N]")
ax[1].set_xlim(0,8)
plt.show()

## 3. A noisy pressure sensor and derived depth

Pressure is measured; depth is derived using estimated environmental parameters.

In [ ]:
rng=np.random.default_rng(18)
sample_t=np.arange(0,180.01,1/60)  # one sample per second
z_sample=synthetic_depth(sample_t)
p_measured=ambient_pressure_pa(z_sample)+300+rng.normal(0,220,len(sample_t))
z_derived=(p_measured-p0)/(rho*g)

def lowpass(x,alpha=0.10):
    y=np.empty_like(x); y[0]=x[0]
    for k in range(1,len(x)): y[k]=alpha*x[k]+(1-alpha)*y[k-1]
    return y

z_est=lowpass(z_derived)
descent_rate=np.gradient(z_est,sample_t*60)
fig,ax=plt.subplots(2,1,sharex=True,figsize=(9,6))
ax[0].plot(sample_t,z_sample,label="true"); ax[0].plot(sample_t,z_est,label="estimated")
ax[0].invert_yaxis(); ax[0].set_ylabel("Depth [m]"); ax[0].legend()
ax[1].plot(sample_t,descent_rate); ax[1].set(xlabel="Time [min]",ylabel="Downward speed [m/s]")
ax[1].set_xlim(0,35); plt.show()

## 4. Remaining gas as a resource state

Surface-equivalent respiratory demand is pressure-scaled and integrated. The assumed workload is deliberately visible.

In [ ]:
surface_rate=18.0
workload=1+0.45*((sample_t>=10)&(sample_t<14))
gas_rate=surface_rate*workload*ambient_pressure_pa(np.maximum(z_est,0))/p0
dt_min=np.diff(sample_t)
used=np.concatenate([[0],np.cumsum((gas_rate[1:]+gas_rate[:-1])*dt_min/2)])
remaining=3000-used

fig,ax=plt.subplots(2,1,sharex=True,figsize=(9,6))
ax[0].plot(sample_t,gas_rate); ax[0].set_ylabel("Demand [surface L/min]")
ax[1].plot(sample_t,remaining); ax[1].set(xlabel="Time [min]",ylabel="Modelled gas [surface L]")
ax[1].set_xlim(0,40); plt.show()
assert np.all(np.diff(remaining)<=1e-10)

## 5. Hidden inert-gas states

The same estimated pressure history drives a bank of educational nitrogen compartments. No ceiling or schedule is calculated.

In [ ]:
f_n2,p_h2o=0.79,0.0627
half_times=np.array([5.,20.,80.,240.])
inspired=f_n2*(ambient_pressure_bar(np.maximum(z_est,0))-p_h2o)
tissue=np.empty((len(half_times),len(sample_t)))
tissue[:,0]=f_n2*(ambient_pressure_bar(0)-p_h2o)
k=np.log(2)/half_times
for j,dt in enumerate(np.diff(sample_t)):
    tissue[:,j+1]=inspired[j]+(tissue[:,j]-inspired[j])*np.exp(-k*dt)

for i,h in enumerate(half_times): plt.plot(sample_t,tissue[i],label=f"{h:g} min")
plt.plot(sample_t,inspired,"k--",alpha=.5,label="inspired N$_2$")
plt.xlabel("Time [min]"); plt.ylabel("Nitrogen pressure [bar]")
plt.title("Different states remember different portions of history"); plt.legend(); plt.show()

## 6. Integrated system dashboard

In [ ]:
fig,ax=plt.subplots(4,1,sharex=True,figsize=(10,10))
ax[0].plot(sample_t,z_est); ax[0].invert_yaxis(); ax[0].set_ylabel("Depth [m]")
ax[1].plot(sample_t,ambient_pressure_bar(np.maximum(z_est,0))); ax[1].set_ylabel("Pressure [bar]")
ax[2].plot(sample_t,remaining); ax[2].set_ylabel("Gas [surface L]")
for i,h in enumerate(half_times): ax[3].plot(sample_t,tissue[i],label=f"{h:g} min")
ax[3].set(xlabel="Time [min]",ylabel="N$_2$ [bar]"); ax[3].legend(ncol=4)
fig.suptitle("One depth history, several coupled subsystem responses")
plt.tight_layout(); plt.show()

## 7. What the dashboard does—and does not—mean

- Pressure follows depth almost immediately.
- Flexible gas volume changes algebraically with pressure.
- Remaining gas carries the integral of past demand.
- Compartment states retain pressure history at several time scales.
- Sensor error propagates into every model driven by estimated depth.

The dashboard does **not** demonstrate that the synthetic profile is safe, that the remaining gas is adequate, or that a real diver would follow these predictions.

## Capstone exercises

1. Classify every variable as state, input, disturbance, parameter, measurement, or derived output.
2. Add uncertainty bands for breathing demand and sensor bias.
3. Compare two depth histories with the same maximum depth and duration.
4. Couple the vertical-motion plant from Notebook 10 instead of prescribing depth.
5. Write a validation matrix stating what evidence each subsystem would require.

## Final reflection

You began with depth and pressure. You can now trace one depth change through gas compression, buoyancy, motion, sensing, estimation, control, resource depletion, and inert-gas state. The durable skill is not memorizing the dashboard; it is knowing what is connected, what is hidden, what is assumed, and what still requires evidence.